<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

In [ ]:
# %load_ext autoreload
# %autoreload 2

import json, yaml
from typing import List, Dict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import sys
import os

import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

In [ ]:
if "google.colab" in sys.modules :
    REPO_PATH = Path("/content/NLP_semeval26_task3_DimASR")

    if not REPO_PATH.exists():
        !git clone "https://github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"
    else :
        !git pull

    %cd "/content/NLP_semeval26_task3_DimASR"
    !git checkout colab_outputs
    sys.path.insert(0, str(REPO_PATH))

from src.data import *
from src.eval import *
from src.models.svr import run_svr_baseline
from src.models.bert import TransformerVARegressor
from src.models.ensemble import (
    AverageEnsemble
)

Cloning into 'NLP_semeval26_task3_DimASR'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 93 (delta 38), reused 75 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 493.04 KiB | 9.86 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/NLP_semeval26_task3_DimASR


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Will be using {device} device.")
# device = torch.device("cpu") # force

# Set testing filter
# for faster training testing
testing = (device.type == "cpu")
if testing :
    print(f"Will be using a lighter training configuration, not suitable for final results.")

Will be using cpu device.
Will be using a lighter training configuration, not suitable for final results.


### Step 1: Load datasets and configuration


In [ ]:
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "restaurant"

!pwd
path = Path(f"data/augmented_{lang}_{domain}_train_alltasks.jsonl")
if path.exists() and not testing : # kill switch
    print("Will be using local augmented dataset")
    train_raw = load_jsonl(f"data/augmented_{lang}_{domain}_train_alltasks.jsonl")
else :
    print("Will be using remote default dataset")
    train_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
                 f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl")
    train_raw = load_jsonl_url(train_url)

predict_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
               f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl")
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

train_df = train_df.sample(100) if testing else train_df

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)


with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

models = config["models"]
print(json.dumps(models, indent=2))

/content/NLP_semeval26_task3_DimASR
Will be using remote default dataset
[
  {
    "name": "distilbert-base-uncased-finetuned-sst-2-english",
    "nickname": "baby_bert_1",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 1,
    "batch_size": 32,
    "dropout": 0.1,
    "max_len": 128
  },
  {
    "name": "distilbert-base-uncased-finetuned-sst-2-english",
    "nickname": "baby_bert_2",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 1,
    "batch_size": 32,
    "dropout": 0.2,
    "max_len": 128
  }
]


### Display the dataframe

In [7]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_restaurant train_df

,Aspect,ID,Text,Valence,Arousal
2987,indian place,rest16_quad_train_1104,i noticed alot of indian people eatting there ...,7.88,7.88
2026,waiters,rest16_quad_train_520,the food is so cheap and the waiters are nice .,7.62,7.75
2626,chicken,rest16_quad_train_877,"stick with the chicken , beef , and lamb dishes .",6.50,6.38
2768,place,rest16_quad_train_964,"this place , which is only a few months old , ...",6.75,6.75
2342,bagels,rest16_quad_train_708,"bagels are ok , but be sure not to make any sp...",5.83,5.83


### subtask_1_eng_restaurant dev_df

,Aspect,ID,Text,Valence,Arousal
1907,service,rest16_quad_train_440,the service was excellent - friendly and atten...,7.67,7.33
3295,NULL,rest16_quad_train_1295,and even more so unpleasant because it was so ...,1.67,8.17
2902,food,rest16_quad_train_1045,not only is the food,5.00,5.00
986,artwork,rest16_quad_test_419,it was romantic - and even nice even with my s...,5.00,5.00
3306,waiter,rest16_quad_train_1304,after the 4th time i asked again and the waite...,4.50,5.12


### subtask_1_eng_restaurant predict_df

,Aspect,VA,ID,Text,Valence,Arousal
0,diner food,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
1,breakfast,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
2,food,7.50#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.75
3,drinks,7.50#7.50,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.50
4,service,7.75#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.75,7.75


### Step 2 : Train all models in config.yaml

In [8]:
model_results = {} # Pour stocker les scores finaux
trained_models = {}
ensemble = dev_df

for arch in models:
    current_model = arch["name"]
    model_type = arch["type"]
    current_nickname = arch.get("nickname")

    print(f"\n{'='*80}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*80}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = float(arch["lr"])
        current_epochs = arch["epochs"]
        current_batch_size = arch["batch_size"]
        current_dropout = arch["dropout"]

        tokenizer = AutoTokenizer.from_pretrained(current_model)
        current_max_len = int(arch.get("max_len", tokenizer.model_max_length))

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création des DataLoaders
        train_dataset = VADataset(train_df, tokenizer, max_len=current_max_len)
        dev_dataset = VADataset(dev_df, tokenizer, max_len=current_max_len)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            print(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score
        trained_models[current_nickname] = model

        # Saving predictions for ensemble learning
        ensemble = predict_to_dataframe(
            model, dev_loader, ensemble,
            pred_v_col = f"{current_nickname}_valence",
            pred_a_col = f"{current_nickname}_arousal"
        )

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = arch["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score

        # ensemble = predict_to_dataframe(
        #     model, dev_loader, ensemble,
        #     pred_v_col = f"{current_model}_valence",
        #     pred_a_col = f"{current_model}_arousal"
        # )


ENTRAÎNEMENT DU MODÈLE : distilbert-base-uncased-finetuned-sst-2-english


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/1 | Train Loss: 43.7178 | Val Loss: 31.2297

ENTRAÎNEMENT DU MODÈLE : distilbert-base-uncased-finetuned-sst-2-english
Paramètres : LR=2e-05, Epochs=1, Batch=32, Dropout=0.2


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/1 | Train Loss: 40.7614 | Val Loss: 28.9486


In [9]:
# torch.save(model.state_dict(), "outputs/checkpoints/distillbert_test.pt")
# temp to avoid retraining, needs to be put into training function

path = "outputs/results/preds.csv"
ensemble.to_csv(path)

ensemble_model = AverageEnsemble(path)
pred_v, pred_a, gold_v, gold_a = ensemble_model.predictions()

eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
model_results["average_ensemble"] = eval_score
trained_models["average_ensemble"] = ensemble_model


### Step 3 : Analyze results

In [10]:
with open("./outputs/results/metrics.yaml", "w") as f:
    # yaml.safe_dump(model_results, f)
    pass

In [11]:
log_path = Path("outputs/results/log.txt")
log_path.parent.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(log_path, "a") as f:
    title = f"\n[{timestamp}] RÉCAPITULATIF DES RÉSULTATS\n"
    f.write(title) ; print(title)
    if "google.colab" in sys.modules :
        f.write(f"Running in Colab with {device} device...")
    for mod, scores in model_results.items():
        line = (
            f"- {mod} : "
            f"PCC_V = {scores['PCC_V']:.4f} | "
            f"PCC_A = {scores['PCC_A']:.4f} | "
            f"RMSE_V = {scores['RMSE_V']:.4f} | "
            f"RMSE_A = {scores['RMSE_A']:.4f}\n"
        )
        print(line) ; f.write(line)


[2026-04-12 01:21:55] RÉCAPITULATIF DES RÉSULTATS

- baby_bert_1 : PCC_V = 0.7927 | PCC_A = -0.1847 | RMSE_V = 4.8637 | RMSE_A = 6.2293

- baby_bert_2 : PCC_V = -0.6133 | PCC_A = 0.2604 | RMSE_V = 4.9182 | RMSE_A = 5.8059

- average_ensemble : PCC_V = 0.6625 | PCC_A = 0.0353 | RMSE_V = 4.8807 | RMSE_A = 6.0159



In [ ]:
# CTRL+S to commit main.ipynb and...
if "google.colab" in sys.modules :
  from google.colab import userdata
  from getpass import getpass

  # GitHub / Settings / Emails (look for 123+user@users.noreply.github.com)
  try:
    email = userdata.get("GITHUB_EMAIL")
  except Exception:
    email = input("Enter your email: ")
  !git config --global user.email {email}

  try:
    name = userdata.get("GITHUB_NAME")
  except Exception:
    name = input("Enter your email: ")
  !git config --global user.name {name}

  !git status
  print()

  !git add outputs/
  !git commit -m "feat: colab outputs"
  print()

  # GitHub / Settings / Developer settings / Personal access tokens
  try:
    token = userdata.get("GITHUB_TOKEN")
  except Exception:
    token = getpass("Enter GitHub token: ")
  !git push https://{token}@github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   outputs/results/log.txt
	modified:   outputs/results/preds.csv

